In [1]:
import torch
from torch.utils.data import DataLoader, ConcatDataset
import torch.nn as nn
import os
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime
from torchinfo import summary
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt 
import numpy as np 
from utils.features import getFeatures
from utils.timeseriesdataset import TimeSeriesDataset
from utils.pad_batch import pad_batch, LABEL_PADDING_VALUE
from models.AlphaModel import AlphaModel
import pickle 
from pathlib import Path

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
torch.cuda.empty_cache()
print('The model is running on:', DEVICE) 

BATCH_SIZE = 32
EPOCHS = 50

The model is running on: cuda:0


# Create DataLoaders

In [2]:
train_instances = []
val_instances = []
test_instances = []

train_files = list(Path("../data/simulated_tracks").glob("*/train_instances.pkl"))
val_files = list(Path("../data/simulated_tracks").glob("*/val_instances.pkl"))
test_files = list(Path("../data/simulated_tracks").glob("*/test_instances.pkl"))

for file in train_files:
    with open(file, "rb") as f:
        train_instances += pickle.load(f)

for file in val_files:
    with open(file, "rb") as f:
        val_instances += pickle.load(f)

for file in test_files:
    with open(file, "rb") as f:
        test_instances += pickle.load(f)

print("Train data: ", len(train_instances), "Test data: ", len(test_instances), "Val data: ", len(val_instances))

Train data:  3187073 Test data:  682945 Val data:  682944


In [3]:
conc_train = ConcatDataset(train_instances)
conc_val = ConcatDataset(val_instances)
conc_test = ConcatDataset(test_instances)

train_loader = DataLoader(conc_train, batch_size=BATCH_SIZE, shuffle=True, collate_fn=pad_batch)
test_loader = DataLoader(conc_test, batch_size=BATCH_SIZE, shuffle=True, collate_fn=pad_batch)
val_loader = DataLoader(conc_val, batch_size=BATCH_SIZE, shuffle=True, collate_fn=pad_batch)

print("DataLoader Sizes:", len(train_loader), len(test_loader), len(val_loader))

# DATA_PATH = "../data/simulated_tracks"
# filepaths = list(Path(DATA_PATH).rglob('*.parquet'))
# random.shuffle(filepaths)

# print("Number of files found:", len(filepaths))

# train_data = []
# test_data = []
# val_data = []

# train_data = [TimeSeriesDataset(filepath, augment=True) for filepath in filepaths[:int(len(filepaths)*0.7)]]
# test_data = [TimeSeriesDataset(filepath, augment=False) for filepath in filepaths[int(len(filepaths)*0.7):int(len(filepaths)*0.85)]]
# val_data = [TimeSeriesDataset(filepath, augment=False) for filepath in filepaths[int(len(filepaths)*0.85):]]

DataLoader Sizes: 99597 21343 21342


# Model
Load the model, optimizer, scheduler, loss


In [4]:
model = AlphaModel().to(DEVICE)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
writer = SummaryWriter('models/checkpoints/alpha_runs/runs_{}'.format(timestamp))
model_directory = os.path.join('models/checkpoints/alpha_model', 'model_{}'.format(timestamp))

print(summary(model, input_size=(BATCH_SIZE, 200, 10)))

continuous_loss_fn = nn.L1Loss(reduction='none')
best_val_loss = float("inf")

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=2e-6)
scheduler = ReduceLROnPlateau(optimizer, factor=0.1, patience=3)

Layer (type:depth-idx)                   Output Shape              Param #
AlphaModel                               [32, 200, 1]              --
├─LayerNorm: 1-1                         [32, 200, 10]             20
├─LSTM: 1-2                              [32, 200, 128]            138,240
├─LSTM: 1-3                              [32, 200, 128]            203,776
├─LayerNorm: 1-4                         [32, 200, 266]            532
├─LSTM: 1-5                              [32, 200, 128]            169,984
├─Linear: 1-6                            [32, 200, 1]              395
Total params: 512,947
Trainable params: 512,947
Non-trainable params: 0
Total mult-adds (Units.GIGABYTES): 3.28
Input size (MB): 0.26
Forward/backward pass size (MB): 33.84
Params size (MB): 2.05
Estimated Total Size (MB): 36.15


# Training Functions

In [5]:
def train_one_epoch(model, optimizer, dataloader):
    model.train()
    running_loss = 0
    runs = 0

    for inputs, alpha_labels,_,_ in dataloader:

        inputs, alpha_labels = inputs.to(DEVICE), alpha_labels.to(DEVICE)
        mask = (alpha_labels != LABEL_PADDING_VALUE).float()

        outputs = model(inputs)
        outputs = outputs.squeeze(-1)
        total_loss = (continuous_loss_fn(outputs, alpha_labels) * mask).sum() / mask.sum()
                
        optimizer.zero_grad()
        total_loss.backward()
        # torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1)
        optimizer.step()
        
        running_loss += total_loss.item()
        runs += 1

        progress_bar.update()

    return running_loss/runs

def evaluate_model(model, dataloader):
    model.eval()
    
    running_val_total = 0.0
    val_runs = 0

    with torch.no_grad():
        for inputs, alpha_labels,_,_ in dataloader:
            
            inputs, alpha_labels = inputs.to(DEVICE), alpha_labels.to(DEVICE)
            mask = (alpha_labels != LABEL_PADDING_VALUE).float()
            
            outputs = model(inputs)  
            outputs = outputs.squeeze(-1)
            total_loss = (continuous_loss_fn(outputs, alpha_labels) * mask).sum() / mask.sum()            
            running_val_total += total_loss.item()
            val_runs += 1
    
    return running_val_total / val_runs

# Train

In [6]:
os.makedirs(model_directory, exist_ok=True)

for epoch in range(EPOCHS):
    print('EPOCH {}:'.format(epoch + 1))

    progress_bar = tqdm(total=len(train_loader), desc='Training', position=0)

    avg_training_loss = train_one_epoch(model, optimizer, train_loader)
    val_total_loss  = evaluate_model(model, val_loader)
    
    print(f'Training LOSS: Alpha {avg_training_loss}\n'
          f'Validation LOSS: Alpha {val_total_loss} \n')
    
    writer.add_scalars('Losses', {
        'Training Alpha Loss': avg_training_loss,
        'Validation Alpha Loss': val_total_loss,
        }, epoch + 1)

    writer.flush()
    
    if val_total_loss < best_val_loss:
        best_val_loss = val_total_loss
        best_model_path = os.path.join(model_directory, f'model_{epoch + 1}')
        torch.save(model.state_dict(), best_model_path)

    scheduler.step(val_total_loss)
    
progress_bar.close()
writer.close()

EPOCH 1:


Training: 100%|██████████| 99597/99597 [37:50<00:00, 44.68it/s]

Training LOSS: Alpha 0.10037804236241799
Validation LOSS: Alpha 0.08441301987270543 

EPOCH 2:


Training: 100%|██████████| 99597/99597 [37:46<00:00, 43.92it/s]

Training LOSS: Alpha 0.09088739312996177
Validation LOSS: Alpha 0.08350844379060969 

EPOCH 3:


...

Training: 100%|██████████| 99597/99597 [37:24<00:00, 44.67it/s]

Training LOSS: Alpha 0.08988062999232543
Validation LOSS: Alpha 0.0828574037144615 

EPOCH 5:


Training: 100%|██████████| 99597/99597 [37:16<00:00, 43.02it/s]

Training LOSS: Alpha 0.08993707488932896
Validation LOSS: Alpha 0.08172188293429343 

EPOCH 6:


Training: 100%|██████████| 99597/99597 [37:32<00:00, 45.45it/s]

In [ ]:
print("Best Validation Loss:", best_val_loss)
print("Best Model Path", best_model_path)

# Testing

In [ ]:
model.load_state_dict(torch.load(best_model_path))
model.eval()

running_test_total = 0.0
test_runs = 0.0

predictions = []
ground_truth = []

progress_bar = tqdm(total=len(test_loader), desc='Testing', position=0)

with torch.no_grad():
    for inputs, alpha_labels,_,_ in test_loader:
        
        inputs, alpha_labels = inputs.to(DEVICE), alpha_labels.to(DEVICE)

        mask = (alpha_labels != LABEL_PADDING_VALUE).float()
        outputs = model(inputs).squeeze(-1)
        total_loss = (continuous_loss_fn(outputs, alpha_labels) * mask).sum() / mask.sum()
        
        running_test_total += total_loss.item()
        test_runs += 1

        predictions.extend(outputs.cpu().numpy())
        ground_truth.extend(alpha_labels.cpu().numpy())
        progress_bar.update()


# Calculate average losses
avg_test_loss = running_test_total / test_runs
print(f'Average test loss: {avg_test_loss}')
progress_bar.close()

# Plot Predictions

In [ ]:
INDEX = 53

padding_starts = (ground_truth[INDEX] == LABEL_PADDING_VALUE).argmax() 

if padding_starts == 0:
    padding_starts = 200

pred_alpha = predictions[INDEX][:padding_starts]
true_alpha = ground_truth[INDEX][:padding_starts]

plt.scatter([i for i in range(len(pred_alpha))], pred_alpha, color="red")
plt.scatter([i for i in range(len(true_alpha))], true_alpha, color="blue")
plt.title("Alpha")
plt.show()